##Video Swin Experiment

## Cell 1 — Mount Drive and create folder structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = '/content/drive/MyDrive/cs_lab'  # Account 1 owns this folder

dirs = ['weights', 'results', 'data/labels', 'data/videos', 'logs']
for d in dirs:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

WEIGHT_PATH  = f'{BASE}/weights/swin_b_k400.pth'
VIDEO_DIR    = f'{BASE}/data/videos'
LABEL_DIR    = f'{BASE}/data/labels'
RESULTS_DIR  = f'{BASE}/results'

print('Folder structure ready')
print('BASE:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder structure ready
BASE: /content/drive/MyDrive/cs_lab


## Cell 2 — Install dependencies

In [ ]:
!pip install -q timm einops pynvml

import torch
print('Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — change runtime to T4')

Torch: 2.11.0+cu128
GPU: Tesla T4


## Cell 3 — Download Swin-B K400 weights

In [ ]:
import hashlib, os, torch
from torchvision.models.video import swin3d_b, Swin3D_B_Weights

if os.path.exists(WEIGHT_PATH) and os.path.getsize(WEIGHT_PATH) > 100e6:
    print('Weights already on Drive, skipping download')
else:
    print('Downloading Swin3D-B K400 weights via torchvision...')
    model_tmp = swin3d_b(weights=Swin3D_B_Weights.KINETICS400_V1)
    torch.save(model_tmp.state_dict(), WEIGHT_PATH)
    del model_tmp
    print('Saved to Drive')

# Verify
with open(WEIGHT_PATH, 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()
print(f'Size: {os.path.getsize(WEIGHT_PATH)/1e6:.1f} MB')
print(f'MD5:  {md5}')
print('Expected MD5: fde19a085812cbb38ecd1b4b1bf63461')
print('Match:', md5 == 'fde19a085812cbb38ecd1b4b1bf63461')

Weights already on Drive, skipping download
Size: 381.8 MB
MD5:  fde19a085812cbb38ecd1b4b1bf63461
Expected MD5: fde19a085812cbb38ecd1b4b1bf63461
Match: True


## Cell 4 — Download SSv2 dataset from Kaggle and copy to Drive

In [ ]:
import os, shutil

# Download to local Colab storage first (faster)
if not os.path.exists('./ssv2_test'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_f9d37c045a507fae3be32a352f2db4af'
    !kaggle datasets download -d ipythonx/ssv2test --unzip -p ./ssv2_test
    print('Downloaded!')
else:
    print('Already downloaded locally')

# Copy to Drive so Account 2 can access it
if not os.path.exists(VIDEO_DIR) or len(os.listdir(VIDEO_DIR)) == 0:
    print('Copying to Drive... takes ~5 min')
    shutil.copytree('./ssv2_test', VIDEO_DIR)
    print('Done')
else:
    print('Videos already on Drive')

classes = [d for d in os.listdir(VIDEO_DIR) if os.path.isdir(f'{VIDEO_DIR}/{d}')]
print(f'Classes on Drive: {len(classes)}')

Dataset URL: https://www.kaggle.com/datasets/ipythonx/ssv2test
License(s): unknown
100% 2.12G/2.12G [00:31<00:00, 71.3MB/s]

Downloaded!
Videos already on Drive
Classes on Drive: 174


## Cell 5 — Create dataset split index files (10 / 20 / 50 / 174 classes)
Saved to shared Drive. Both notebooks use the same JSON files.

In [ ]:
import os, random, json

classes = sorted([d for d in os.listdir(VIDEO_DIR) if os.path.isdir(f'{VIDEO_DIR}/{d}')])

all_data = []
for cls in classes:
    for vid in os.listdir(f'{VIDEO_DIR}/{cls}'):
        all_data.append({'path': f'{VIDEO_DIR}/{cls}/{vid}', 'label': cls})

print(f'Total videos: {len(all_data)}')

# Incremental subsets: 10 ⊂ 20 ⊂ 50 ⊂ 174
random.seed(42)
all_shuffled = random.sample(classes, len(classes))
selected_classes = {
    10:  all_shuffled[:10],
    20:  all_shuffled[:20],
    50:  all_shuffled[:50],
    174: all_shuffled
}

for n, cls_list in selected_classes.items():
    path = f'{LABEL_DIR}/split_{n}class.json'
    if os.path.exists(path):
        print(f'{n} classes: already exists, skipping')
        continue
    subset = [d for d in all_data if d['label'] in cls_list]
    random.seed(42)
    random.shuffle(subset)
    split_idx = int(len(subset) * 0.8)
    out = {'classes': cls_list, 'train': subset[:split_idx], 'val': subset[split_idx:]}
    with open(path, 'w') as f:
        json.dump(out, f)
    print(f'{n} classes: {len(subset)} videos ({len(out["train"])} train, {len(out["val"])} val) -> saved')

Total videos: 27157
10 classes: already exists, skipping
20 classes: already exists, skipping
50 classes: already exists, skipping
174 classes: already exists, skipping


## Cell 6 — Config

In [ ]:
CONFIG = {
    'epochs':       5,
    'batch_size':   8,
    'num_frames':   8,    # Video Swin native
    'img_size':     224,
    'lr':           1e-4,
    'lora_r':       8,
    'lora_alpha':   16,
    'lora_dropout': 0.1,
    'splits':       [10, 20, 50, 174],
    'seed':         42
}

REFERENCE = """
Hu et al., LoRA: Low-Rank Adaptation of Large Language Models, ICLR 2022.
https://arxiv.org/abs/2106.09685
GitHub: https://github.com/YOUR_USERNAME/YOUR_REPO
"""

print('CONFIG ready')
print(CONFIG)

CONFIG ready
{'epochs': 5, 'batch_size': 8, 'num_frames': 8, 'img_size': 224, 'lr': 0.0001, 'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1, 'splits': [10, 20, 50, 174], 'seed': 42}


## Cell 7 — Dataset class
Loads videos frame by frame, samples num_frames evenly, applies normalization.

In [ ]:
import cv2, json, torch, numpy as np
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

class SSV2Dataset(Dataset):
    def __init__(self, split_json, split='train', num_frames=8, img_size=224):
        with open(split_json) as f:
            data = json.load(f)
        self.samples = data[split]
        self.classes = data['classes']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.num_frames = num_frames
        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.samples)

    def _load_video(self, path):
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = np.linspace(0, max(total-1, 0), self.num_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                frame = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(self.transform(frame))
        cap.release()
        return torch.stack(frames)  # (T, C, H, W)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        video  = self._load_video(sample['path'])  # (T, C, H, W)
        video  = video.permute(1, 0, 2, 3)          # (C, T, H, W)
        label  = self.class_to_idx[sample['label']]
        return video, label

# Quick test
ds = SSV2Dataset(f'{LABEL_DIR}/split_10class.json', 'train',
                 CONFIG['num_frames'], CONFIG['img_size'])
v, l = ds[0]
print(f'Video shape: {v.shape}')  # (3, 8, 224, 224)
print(f'Label: {l}')
print(f'Dataset size: {len(ds)}')

Video shape: torch.Size([3, 8, 224, 224])
Label: 8
Dataset size: 1148


## Cell 8 — Metrics logger
Tracks latency, GPU power, peak memory, and energy = power x latency.

In [ ]:
import time, pynvml, numpy as np, torch

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def get_power_watts():
    return pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0

class MetricsLogger:
    def reset(self):
        self.power_readings = []
        self.start_time = None

    def start(self):
        torch.cuda.reset_peak_memory_stats()
        self.power_readings = []
        self.start_time = time.time()

    def sample_power(self):
        self.power_readings.append(get_power_watts())

    def stop(self):
        elapsed   = time.time() - self.start_time
        avg_power = float(np.mean(self.power_readings)) if self.power_readings else 0.0
        peak_mem  = torch.cuda.max_memory_allocated() / 1e6
        energy    = avg_power * elapsed
        return {
            'latency_s':      round(elapsed, 2),
            'avg_power_w':    round(avg_power, 2),
            'peak_memory_mb': round(peak_mem, 2),
            'energy_j':       round(energy, 2)
        }

logger = MetricsLogger()
logger.reset()
print('MetricsLogger ready')

MetricsLogger ready


## Cell 9 — LoRA implementation
Applies low-rank adaptation to all Linear layers with in_features > 64.
Only LoRA parameters + classification head are trainable (~1.8% of total params).

In [ ]:
import torch.nn as nn, torch, math

class LoRALinear(nn.Module):
    """
    Wraps a frozen Linear layer and adds a low-rank update: h = Wx + (A @ B) * scaling
    Reference: Hu et al., LoRA, ICLR 2022. https://arxiv.org/abs/2106.09685
    GitHub: https://github.com/YOUR_USERNAME/YOUR_REPO
    """
    def __init__(self, original_linear, r, alpha, dropout=0.1):
        super().__init__()
        self.original = original_linear
        self.scaling  = alpha / r
        in_f, out_f   = original_linear.in_features, original_linear.out_features
        self.lora_A   = nn.Parameter(torch.randn(in_f, r) * 0.01)
        self.lora_B   = nn.Parameter(torch.zeros(r, out_f))
        self.dropout  = nn.Dropout(dropout)
        # Freeze original weights
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False

    # Expose original attributes so model internals don't break
    @property
    def weight(self):      return self.original.weight
    @property
    def bias(self):        return self.original.bias
    @property
    def in_features(self): return self.original.in_features
    @property
    def out_features(self):return self.original.out_features

    def forward(self, x):
        return self.original(x) + self.dropout(x) @ self.lora_A @ self.lora_B * self.scaling


def apply_lora(model, r=8, alpha=16, dropout=0.1):
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.in_features > 64:
            if '.' not in name:  # skip top-level modules
                continue
            parent_name, child_name = name.rsplit('.', 1)
            parent = dict(model.named_modules())[parent_name]
            setattr(parent, child_name, LoRALinear(module, r, alpha, dropout))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'LoRA applied — trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M ({100*trainable/total:.1f}%)')
    return model

print('LoRA ready')

LoRA ready


## Cell 10 — Training function
Runs fine-tuning + inference for one split, logs all metrics.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader

def train_one_split(model, split_n, num_frames, model_name, ft_type, config):
    split_json   = f'{LABEL_DIR}/split_{split_n}class.json'
    train_ds     = SSV2Dataset(split_json, 'train', num_frames, config['img_size'])
    val_ds       = SSV2Dataset(split_json, 'val',   num_frames, config['img_size'])
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=config['batch_size'],
                              shuffle=False, num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=config['lr'])
    criterion = nn.CrossEntropyLoss()

    # Fine-tuning phase
    model.train()
    logger.reset()
    logger.start()
    for epoch in range(config['epochs']):
        for videos, labels in train_loader:
            videos, labels = videos.cuda(), labels.cuda()
            optimizer.zero_grad()
            loss = criterion(model(videos), labels)
            loss.backward()
            optimizer.step()
            logger.sample_power()
        print(f'  Epoch {epoch+1}/{config["epochs"]} done')
    ft_metrics = logger.stop()

    # Inference phase
    model.eval()
    logger.reset()
    logger.start()
    correct = total = 0
    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.cuda(), labels.cuda()
            preds   = model(videos).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            logger.sample_power()
    inf_metrics = logger.stop()

    result = {
        'model':            model_name,
        'ft_type':          ft_type,
        'split':            split_n,
        'accuracy':         round(correct / total * 100, 2),
        'ft_latency_s':     ft_metrics['latency_s'],
        'ft_avg_power_w':   ft_metrics['avg_power_w'],
        'ft_peak_memory_mb':ft_metrics['peak_memory_mb'],
        'ft_energy_j':      ft_metrics['energy_j'],
        'inf_latency_s':    inf_metrics['latency_s'],
        'inf_avg_power_w':  inf_metrics['avg_power_w'],
        'inf_peak_memory_mb':inf_metrics['peak_memory_mb'],
        'inf_energy_j':     inf_metrics['energy_j'],
    }
    print(f'Result: {result}')
    return result

print('Training function ready')

Training function ready


## Cell 11 — Run all experiments (Video Swin)
Runs Full FT and LoRA for each split. Saves results after every split.

In [ ]:
import json, os, torch
from torchvision.models.video import swin3d_b, Swin3D_B_Weights

"""
Reference: Hu et al., LoRA, ICLR 2022. https://arxiv.org/abs/2106.09685
GitHub: https://github.com/YOUR_USERNAME/YOUR_REPO
"""

results = []
out_path = f'{RESULTS_DIR}/results_videoswin.json'

for split_n in CONFIG['splits']:
    print(f'\n{"="*50}')
    print(f'Split: {split_n} classes')

    # --- Full Fine-Tuning ---
    print('\n[1/2] Full Fine-Tuning')
    model_ft = swin3d_b(weights=Swin3D_B_Weights.KINETICS400_V1)
    in_features = model_ft.head.in_features
    model_ft.head = torch.nn.Linear(in_features, split_n)
    for p in model_ft.parameters():
        p.requires_grad = True
    model_ft = model_ft.cuda()

    r = train_one_split(model_ft, split_n, CONFIG['num_frames'], 'VideoSwin', 'FullFT', CONFIG)
    results.append(r)
    del model_ft
    torch.cuda.empty_cache()

    # --- LoRA Fine-Tuning ---
    print('\n[2/2] LoRA Fine-Tuning')
    model_lora = swin3d_b(weights=Swin3D_B_Weights.KINETICS400_V1)
    model_lora.head = torch.nn.Linear(in_features, split_n)
    for p in model_lora.parameters():
        p.requires_grad = False
    model_lora = apply_lora(model_lora, r=CONFIG['lora_r'],
                             alpha=CONFIG['lora_alpha'], dropout=CONFIG['lora_dropout'])
    for p in model_lora.head.parameters():
        p.requires_grad = True
    model_lora = model_lora.cuda()

    r = train_one_split(model_lora, split_n, CONFIG['num_frames'], 'VideoSwin', 'LoRA', CONFIG)
    results.append(r)
    del model_lora
    torch.cuda.empty_cache()

    # Save after every split
    with open(out_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Saved {len(results)} results so far -> {out_path}')

print('\nAll Video Swin experiments complete!')


Split: 10 classes

[1/2] Full Fine-Tuning
Downloading: "https://download.pytorch.org/models/swin3d_b_1k-24f7c7c6.pth" to /root/.cache/torch/hub/checkpoints/swin3d_b_1k-24f7c7c6.pth


100%|██████████| 364M/364M [00:07<00:00, 50.1MB/s]


  Epoch 1/5 done
  Epoch 2/5 done
  Epoch 3/5 done
  Epoch 4/5 done
  Epoch 5/5 done
Result: {'model': 'VideoSwin', 'ft_type': 'FullFT', 'split': 10, 'accuracy': 68.29, 'ft_latency_s': 1100.66, 'ft_avg_power_w': 66.65, 'ft_peak_memory_mb': 10390.26, 'ft_energy_j': 73356.89, 'inf_latency_s': 45.96, 'inf_avg_power_w': 66.64, 'inf_peak_memory_mb': 2571.19, 'inf_energy_j': 3063.07}

[2/2] LoRA Fine-Tuning
LoRA applied — trainable: 1.58M / 89.23M (1.8%)
  Epoch 1/5 done
  Epoch 2/5 done
  Epoch 3/5 done
  Epoch 4/5 done
  Epoch 5/5 done
Result: {'model': 'VideoSwin', 'ft_type': 'LoRA', 'split': 10, 'accuracy': 71.43, 'ft_latency_s': 1054.15, 'ft_avg_power_w': 66.2, 'ft_peak_memory_mb': 8783.87, 'ft_energy_j': 69790.11, 'inf_latency_s': 46.95, 'inf_avg_power_w': 66.89, 'inf_peak_memory_mb': 1522.88, 'inf_energy_j': 3140.82}
Saved 2 results so far -> /content/drive/MyDrive/cs_lab/results/results_videoswin.json

Split: 20 classes

[1/2] Full Fine-Tuning
  Epoch 1/5 done
  Epoch 2/5 done
  Epoc

In [ ]:
import json
import pandas as pd

path = "/content/drive/MyDrive/cs_lab/results/results_videoswin.json"

with open(path, "r") as f:
    results = json.load(f)

pd.DataFrame(results)

,model,ft_type,split,accuracy,ft_latency_s,ft_avg_power_w,ft_peak_memory_mb,ft_energy_j,inf_latency_s,inf_avg_power_w,inf_peak_memory_mb,inf_energy_j
0,VideoSwin,FullFT,10,68.29,1100.66,66.65,10390.26,73356.89,45.96,66.64,2571.19,3063.07
1,VideoSwin,LoRA,10,71.43,1054.15,66.20,8783.87,69790.11,46.95,66.89,1522.88,3140.82
2,VideoSwin,FullFT,20,50.08,2415.06,66.60,10389.75,160838.37,109.65,66.67,2568.62,7310.32
3,VideoSwin,LoRA,20,58.24,2332.66,66.20,8784.00,154432.35,111.11,67.10,1523.05,7455.42


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!find /content/drive -name "results_videoswin.json"

/content/drive/MyDrive/cs_lab/results/results_videoswin.json


In [ ]:
import json
from collections import Counter

split_json = f'{LABEL_DIR}/split_10class.json'

with open(split_json, "r") as f:
    data = json.load(f)

print("Classes:", data["classes"])
print("Num classes:", len(data["classes"]))

print("\nTrain samples:", len(data["train"]))
print("Val samples:", len(data["val"]))

print("\nTrain label distribution:")
print(Counter([s["label"] for s in data["train"]]))

print("\nVal label distribution:")
print(Counter([s["label"] for s in data["val"]]))

Classes: ['Trying to pour something into something, but missing so it spills next to it', 'Lifting something up completely, then letting it drop down', 'Covering something with something', 'Pretending to pour something out of something, but something is empty', 'Pouring something out of something', 'Poking something so that it falls over', 'Moving something across a surface without it falling down', 'Lifting a surface with something on it until it starts sliding down', 'Spinning something so it continues spinning', 'Letting something roll along a flat surface']
Num classes: 10

Train samples: 1148
Val samples: 287

Train label distribution:
Counter({'Covering something with something': 310, 'Spinning something so it continues spinning': 165, 'Letting something roll along a flat surface': 160, 'Lifting something up completely, then letting it drop down': 122, 'Pouring something out of something': 99, 'Poking something so that it falls over': 89, 'Moving something across a surface withou